# Statistics for Data Analysts/Engineers — Module 5: Sampling and the Central Limit Theorem

This is one of the most important modules in the whole course. So far we've worked with
one specific set of 400 customers as if it were our entire knowledge of the world. In
practice, your 400 customers are usually a **sample** drawn from a much larger
**population** (all current and future customers). Inferential statistics -- which we're
starting for real here -- answers the question: what can I say about the WHOLE
population, given only a sample?

## Table of contents
1. [Population vs. sample, parameter vs. statistic](#sec1)
2. [The sampling distribution of the mean](#sec2)
3. [The Central Limit Theorem — a simulation](#sec3)
4. [Standard error](#sec4)
5. [How sample size affects precision](#sec5)
6. [Summary and exercises](#sec6)


Every module in this course starts by generating the same synthetic dataset -- thanks to a fixed random seed (`np.random.default_rng(42)`), the numbers always come out identical, so you can open any module independently of the others, even after restarting the kernel.

In this module we'll treat our set of 400 customers as the **entire population** (we know its true, exact mean), so we can simulate random sampling from it and compare samples against a truth we normally wouldn't know.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

city = rng.choice(
    ["New York", "Chicago", "Austin", "Seattle", "Denver"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["North", "South", "East", "West"], size=n)
age = rng.integers(18, 70, size=n)

# monthly spend: right-skewed distribution (typical for spending data)
monthly_spend = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satisfaction correlated with spend + a region effect (deliberately different) + random
# noise, clipped to a 1-10 scale -- this gives Module 10 (ANOVA) a real between-group gap
region_effect = {"North": 0.0, "South": -0.5, "East": 0.3, "West": 0.8}
satisfaction = np.clip(
    monthly_spend / 45
    + np.array([region_effect[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# A/B campaign group -- with a REAL injected effect on conversion (group B better)
group = rng.choice(["A", "B"], size=n)
conversion_prob = np.where(group == "A", 0.10, 0.20)
converted = rng.binomial(1, conversion_prob)

# spend before/after a loyalty-program rollout (paired data)
spend_before = rng.normal(220, 40, size=n).round(2)
program_effect = rng.normal(15, 10, size=n)
spend_after = (spend_before + program_effect).round(2)

customers = pd.DataFrame({
    "customer_id": np.arange(1, n + 1),
    "city": city,
    "region": region,
    "age": age,
    "monthly_spend": monthly_spend,
    "satisfaction": satisfaction,
    "campaign_group": group,
    "converted": converted,
    "spend_before": spend_before,
    "spend_after": spend_after,
})
customers.head()


<a id="sec1"></a>
## 1. Population vs. sample, parameter vs. statistic

- **Population** — the entire set you actually care about (e.g. all current and future
  customers of a shop). A number describing the population (e.g. its true mean) is
  called a **parameter**, traditionally denoted with a Greek letter, e.g. `mu`
  (population mean).
- **Sample** — a subset of the population you actually measured/collected (e.g. our 400
  customers). A number computed from a sample (e.g. its mean) is called a **statistic**,
  denoted with a Latin letter, e.g. `x-bar` (sample mean).

The key question of inferential statistics: how well does a sample statistic (`x-bar`)
approximate the unknown population parameter (`mu`)? The answer depends on how the sample
was drawn and how large it is.

In [ ]:
population_mu = customers["monthly_spend"].mean()  # in practice you would NOT know this number
print(f"True mean of the entire 'population' (400 customers): {population_mu:.2f}")


> ⚠️ **Sampling must be representative**
>
> All the math in this module assumes **random, representative sampling** -- every member of the population has a known chance of ending up in the sample. If the sample is biased (e.g. you only survey customers who left a review, or only from one city), none of the techniques below fix that -- "garbage in, garbage out". Sampling bias is a topic in its own right that goes beyond this course, but it's worth keeping in mind whenever you collect data.

<a id="sec2"></a>
## 2. The sampling distribution of the mean

Imagine that instead of taking ONE sample of 30 customers, you take thousands of
different 30-customer samples (drawing fresh each time) and compute the mean each time.
You'd end up with thousands of different sample means -- and those means have a
distribution of their own! We call it the **sampling distribution of the mean**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
sample_size = 30
num_samples = 5000

sample_means = [
    rng.choice(customers["monthly_spend"], size=sample_size, replace=False).mean()
    for _ in range(num_samples)
]

plt.figure(figsize=(8, 4))
plt.hist(sample_means, bins=40, edgecolor="white")
plt.axvline(population_mu, color="red", linestyle="--", label=f"True population mean: {population_mu:.1f}")
plt.xlabel("Sample mean (n=30)")
plt.ylabel("Number of samples")
plt.title(f"Sampling distribution of the mean ({num_samples} samples of {sample_size} customers)")
plt.legend()
plt.show()


Notice two things: (1) the distribution of sample means is clearly **centered around
the true population mean** -- even though any single sample might land far from the
truth, over many repeats the means "center themselves"; (2) the shape of this
distribution looks **approximately bell-shaped (normal)** -- even though the original
`monthly_spend` (see Module 2) is clearly right-skewed, nothing like a bell!

<a id="sec3"></a>
## 3. The Central Limit Theorem — a simulation

What we just observed is no accident -- it's the **Central Limit Theorem** (CLT):
regardless of the shape of the population's distribution, the distribution of means from
sufficiently large samples approaches a normal distribution as the sample size (`n`)
grows. This is one of the most important theorems in all of statistics -- without it,
most statistical tests (Modules 7+) simply wouldn't work.

Let's see this with an even more extreme example than spend: we'll draw data from an
exponential distribution (even more strongly skewed) and compare the shape of the
sampling distribution for different sample sizes `n`.

In [ ]:
from scipy import stats

# a "population" defined by us: a strongly right-skewed exponential distribution
exponential_population = stats.expon(scale=100).rvs(size=100_000, random_state=1)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

axes[0].hist(exponential_population, bins=50, color="gray")
axes[0].set_title("Population (exponential distribution)")

for ax, n in zip(axes[1:], [2, 10, 50]):
    means = [
        rng.choice(exponential_population, size=n, replace=False).mean()
        for _ in range(3000)
    ]
    ax.hist(means, bins=40, color="steelblue")
    ax.set_title(f"Sample means, n={n}")

plt.tight_layout()
plt.show()


The population (far left) is very skewed. At `n=2` the distribution of means is still
clearly skewed -- two observations aren't enough to "smooth out" the asymmetry. At
`n=10` the shape already clearly resembles a bell. At `n=50` it's nearly perfectly
symmetric and bell-shaped -- **the CLT in action**.

> 📊 **How large does n need to be?**
>
> A popular rule of thumb says `n >= 30` is usually enough for the CLT to "kick in" well for most distributions seen in practice. The more skewed/unusual the population (like our exponential distribution), the larger `n` you might need. If the population is already fairly normal, even a very small `n` is enough.

<a id="sec4"></a>
## 4. Standard error

Since the sampling distribution of the mean has its own distribution, it also has its
own standard deviation -- called the **standard error** (SE), to distinguish it from the
standard deviation of individual observations. Theoretical formula:

```
SE = sigma / sqrt(n)
```

where `sigma` is the population's standard deviation and `n` is the sample size.

In [ ]:
population_sigma = customers["monthly_spend"].std(ddof=0)
n = 30

theoretical_se = population_sigma / np.sqrt(n)
simulated_se = np.std(sample_means, ddof=1)  # from our simulation in section 2 (n=30)

print(f"Theoretical SE (formula): {theoretical_se:.2f}")
print(f"SE observed in the simulation: {simulated_se:.2f}")


Both numbers should come out very close -- our "brute force" simulation (thousands of
samples) confirms the ready-made mathematical formula, which will let us compute this
without simulating anything from here on.

<a id="sec5"></a>
## 5. How sample size affects precision

The formula `SE = sigma / sqrt(n)` says something very concrete: to **cut the standard
error in half**, you need a **fourfold larger** sample (because of the square root in the
denominator). This is an important practical intuition -- doubling your sample size
doesn't double your precision, it only improves it by a factor of `sqrt(2) ≈ 1.41`.

In [ ]:
sample_sizes = [5, 10, 30, 50, 100, 200]
standard_errors = [population_sigma / np.sqrt(n) for n in sample_sizes]

plt.figure(figsize=(8, 4))
plt.plot(sample_sizes, standard_errors, marker="o")
plt.xlabel("Sample size (n)")
plt.ylabel("Standard error (SE)")
plt.title("SE shrinks with n, but more and more slowly")
plt.show()


The curve flattens out at larger `n` -- the first few dozen extra observations give a
huge precision boost, and the next few hundred give less and less. This is exactly the
same intuition we'll return to in Module 14 when planning sample size for an A/B test:
"more data always helps, but with diminishing returns".

<a id="sec6"></a>
## 6. Summary and exercises

In this module we covered:
- the difference between population/parameter and sample/statistic,
- the sampling distribution of the mean -- the distribution of means from many samples,
- the Central Limit Theorem: sample means approach a normal distribution regardless of
  the population's shape, as `n` grows,
- the standard error `SE = sigma / sqrt(n)` as a measure of a sample mean's precision,
- that improving precision requires a disproportionately larger sample (the square root
  in the denominator).

> 📝 **Exercise 1: CLT for a different column**
>
> Repeat the simulation from section 2 (5000 samples of 30 observations), but for the `satisfaction` column instead of `monthly_spend`. Plot a histogram of the sample means. Does the distribution look approximately normal, even though `satisfaction` itself isn't perfectly symmetric?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
satisfaction_means = [
    rng.choice(customers["satisfaction"], size=30, replace=False).mean()
    for _ in range(5000)
]

plt.figure(figsize=(8, 4))
plt.hist(satisfaction_means, bins=40, edgecolor="white")
plt.xlabel("Sample mean of satisfaction (n=30)")
plt.title("Sampling distribution of the mean satisfaction")
plt.show()
```
</details>

> 📝 **Exercise 2: SE for different columns**
>
> Compute the theoretical standard error (`sigma / sqrt(n)`) for `n=40` separately for the `monthly_spend`, `age`, and `satisfaction` columns. Which column has the largest standard error at the same sample size? What does that say about its spread?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
for column in ["monthly_spend", "age", "satisfaction"]:
    sigma = customers[column].std(ddof=0)
    se = sigma / np.sqrt(40)
    print(f"{column}: sigma={sigma:.2f}, SE(n=40)={se:.3f}")
```

The column with the largest spread (largest sigma) will also have the largest SE at the same sample size -- more variability in the underlying data always translates into a less precise sample mean.
</details>

> 📝 **Exercise 3: How much larger a sample?**
>
> Suppose the current SE for `monthly_spend` at `n=30` is some value X. How much larger a sample do you need to cut the SE exactly in half? Verify computationally (compute SE for `n=30` and `n=120`, compare the ratio).

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
se_30 = population_sigma / np.sqrt(30)
se_120 = population_sigma / np.sqrt(120)
print(f"SE at n=30: {se_30:.3f}")
print(f"SE at n=120: {se_120:.3f}")
print(f"Ratio SE(30)/SE(120): {se_30 / se_120:.2f}")
```

A fourfold increase in n (30 -> 120) should give exactly a twofold decrease in SE, since sqrt(4) = 2 -- a direct illustration of the formula from section 5.
</details>

> 🔥 **Challenge: CLT for the median instead of the mean**
>
> The CLT in its basic form is about the mean, but you can check empirically whether the sampling distribution of the MEDIAN behaves similarly. Repeat the simulation from section 2, but computing the median instead of the mean for each of 5000 `monthly_spend` samples (n=30). Plot a histogram of the sample medians and compare its shape to the histogram of sample means from section 2 -- does it also look roughly bell-shaped?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
sample_medians = [
    np.median(rng.choice(customers["monthly_spend"], size=30, replace=False))
    for _ in range(5000)
]

plt.figure(figsize=(8, 4))
plt.hist(sample_medians, bins=40, edgecolor="white")
plt.xlabel("Sample median (n=30)")
plt.title("Sampling distribution of the median")
plt.show()
```

The distribution of sample medians also approaches a bell shape (though typically wider/less efficient than for the mean) -- CLT-like results exist for many statistics, not just the mean, but the mean has the simplest, most well-known theory.
</details>

## What's next?

Now that we know the sample mean has its own (approximately normal) distribution and we
know its standard error, we can build a **confidence interval** in **Module 6** -- a range
of values that, at a given confidence level, contains the true, unknown population mean.